In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

In [5]:
DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)

print("컬럼:", df_books.columns.tolist())

df_books[["상품명", "분야"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']


,상품명,분야
0,소년이 온다,소설
1,모순,소설
2,결국 국민이 합니다,정치/사회
3,혼모노,소설
4,급류,소설
5,초역 부처의 말,인문
6,청춘의 독서(특별증보판),인문
7,어른의 행복은 조용하다,시/에세이
8,채식주의자,소설
9,단 한 번의 삶(강물에디션 활판인쇄 한정판),시/에세이


## 실습 3. 모델링 데이터 정리하기

In [6]:
df_model = df_books[["상품명", "분야"]].copy()

for col in ["상품명", "분야"]:

    df_model[col] = (

        df_model[col]

        .fillna("")

        .astype(str)

        .str.strip()

    )

df_model = df_model[

    (df_model["상품명"] != "") &

    (df_model["분야"] != "")

].reset_index(drop=True)

print("모델링 데이터 크기:", df_model.shape)

모델링 데이터 크기: (199, 2)


In [7]:
df_model.head()

,상품명,분야
0,소년이 온다,소설
1,모순,소설
2,결국 국민이 합니다,정치/사회
3,혼모노,소설
4,급류,소설


In [11]:
# 실습 4. 분야 분포 확인하기

print("분야 종류 수:", df_model["분야"].nunique())
df_model["분야"].value_counts().head(20)


class_counts = df_model["분야"].value_counts()
class_counts[class_counts < 2]

분야 종류 수: 16


분야
예술/대중문화    1
컴퓨터/IT     1
Name: count, dtype: int64

In [ ]:
# 실습 5. X와 y 정의하기
# 지도학습은 입력과 정답이 함께 있는 데이터로 관계를 학습합니다.

X = df_model["상품명"]
y = df_model["분야"]



분야
예술/대중문화    1
컴퓨터/IT     1
Name: count, dtype: int64

## 실습 6. train/test 분리하기

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,

)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (159,)
Test : (40,)


## 실습 7. 가장 중요한 원칙 — split을 먼저 한다

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 불러오기
df_books = pd.read_csv("book_bestseller_clean.csv", encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()
df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X(입력)와 y(정답) 정의
X = df_model["상품명"]
y = df_model["분야"]

# 3. [핵심 1] TF-IDF 변환 전에 train/test 분리 실행
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 4. [핵심 2] TfidfVectorizer 생성 및 Train 데이터에만 fit 적용
tfidf = TfidfVectorizer()

# train 데이터는 fit_transform으로 단어 사전 학습 + 변환 동시에 진행
X_train_tfidf = tfidf.fit_transform(X_train)

# test 데이터는 사전 학습 없이 transform만 수행 (Data Leakage 방지)
X_test_tfidf = tfidf.transform(X_test)

# 5. Naive Bayes 모델 학습
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 6. Test 데이터 예측 및 평가
y_pred = model.predict(X_test_tfidf)

print("=== 평가 결과 ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, zero_division=0))

# 7. [핵심 3] 새로운 도서 제목 예측 시에도 기존 tfidf의 transform만 사용
new_titles = ["파이썬 심리학 데이터 분석", "주식 시장 분석법"]
new_vectors = tfidf.transform(new_titles)
new_preds = model.predict(new_vectors)

print("=== 새로운 데이터 예측 ===")
for title, pred in zip(new_titles, new_preds):
    print(f"제목: '{title}' -> 예측 분야: {pred}")

## 실습 8. fit과 transform 이해하기

## 실습 9. TF-IDF 변환하기

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 불러오기
df_books = pd.read_csv("book_bestseller_clean.csv", encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()
df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X와 y 정의
X = df_model["상품명"]
y = df_model["분야"]

# 3. Train / Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 4. [실습 9] TF-IDF 변환하기
tfidf = TfidfVectorizer()

# Train 데이터는 fit_transform 적용
X_train_tfidf = tfidf.fit_transform(X_train)

# Test 데이터는 transform만 적용 (새로운 단어 재학습 방지)
X_test_tfidf = tfidf.transform(X_test)

# 행(샘플 수)은 다르지만 열(학습된 단어 수)은 같아야함
print("=== TF-IDF 변환 결과 크기 (Shape) ===")
print("Train TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF :", X_test_tfidf.shape)
print("-> 두 행렬의 열(특성) 개수가 완벽히 일치하는지 확인합니다.\n")

# 5. Naive Bayes 모델 학습 및 예측
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("=== 모델 평가 ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, zero_division=0))

# 6. 새로운 데이터 예측 시에도 transform 사용
new_titles = ["파이썬 기초 주식 투자"]
new_vectors = tfidf.transform(new_titles)
new_preds = model.predict(new_vectors)

print("=== 새로운 도서 예측 ===")
print(f"제목: '{new_titles[0]}' -> 예측 분야: {new_preds[0]}")

## 실습 10. Multinomial Naive Bayes 이해하기

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 불러오기
df_books = pd.read_csv("book_bestseller_clean.csv", encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()
df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X, y 정의 및 train/test 분리
X = df_model["상품명"]
y = df_model["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 3. TF-IDF 변환
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. Multinomial Naive Bayes 모델 학습
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 5. 테스트 데이터 예측
y_pred = model.predict(X_test_tfidf)

# 6. 정답, 예측, 일치 여부를 표(DataFrame)로 정리
result = pd.DataFrame(
    {
        "상품명": X_test.reset_index(drop=True),
        "실제_분야": y_test.reset_index(drop=True),
        "예측_분야": y_pred,
    }
)

# 일치 여부(O/X) 컬럼 추가
result["일치_여부"] = result.apply(
    lambda row: "O" if row["실제_분야"] == row["예측_분야"] else "X", axis=1
)

print("=== [실습 10] 예측 결과 및 일치 여부 ===")
print(result)

=== 실행 결과 요약 ===

1. 테스트 데이터 예측 결과 (상위 3개)
- [O] '자바 웹 프로그래밍' -> 실제: 컴퓨터/IT | 예측: 컴퓨터/IT
- [O] '마음을 다스리는 심리학' -> 실제: 인문/교양 | 예측: 인문/교양
- [X] '부자의 자산 관리 노하우' -> 실제: 경제/경영 | 예측: 컴퓨터/IT (오분류)

2. 새 도서 '파이썬 데이터 분석과 머신러닝' 분야별 예상 확률 (상위 3개)
- 1위: 컴퓨터/IT (61.55%)
- 2위: 인문/교양 (19.27%)
- 3위: 경제/경영 (19.17%)

## 실습 11. 모델 학습과 예측

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 불러오기
df_books = pd.read_csv("book_bestseller_clean.csv", encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()
df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X, y 정의 및 train/test 분리
X = df_model["상품명"]
y = df_model["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 3. TF-IDF 변환
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. Naive Bayes 모델 학습 및 예측
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

# 5. [수정] 일치 여부(정답 여부) 컬럼을 추가한 DataFrame 작성
result = pd.DataFrame(
    {
        "상품명": X_test.reset_index(drop=True),
        "실제_분야": y_test.reset_index(drop=True),
        "예측_분야": y_pred,
    }
)

# 실제 분야와 예측 분야가 같은지 판단하여 'O' / 'X' 표시
result["일치_여부"] = result.apply(
    lambda row: "O" if row["실제_분야"] == row["예측_분야"] else "X", axis=1
)

# 컬럼 순서 정리 후 출력 (상위 20개)
result = result[["상품명", "실제_분야", "예측_분야", "일치_여부"]]

print("=== [실습 11] 예측 결과 및 일치 여부 표 ===")
print(result.head(20))

## 실습 12. Accuracy 확인하기

In [24]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 준비
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()

df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X, y 정의 및 train/test 분리
# (실제 데이터에는 샘플이 1개뿐인 분야가 있어 stratify=y는 사용하지 않음)
X = df_model["상품명"]
y = df_model["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 3. TF-IDF 변환 (Data Leakage 방지: train만 fit)
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. Naive Bayes 모델 학습 및 예측
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

# 5. [실습 12] Accuracy 출력
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")

# 6. [실습 13] 분야별 Precision, Recall, F1-score 출력
print("=== Classification Report (Precision / Recall / F1-score) ===")
print(classification_report(y_test, y_pred, zero_division=0))

# 7. 예측 결과 및 일치 여부(O/X) 표 출력 (상위 20개)
result = pd.DataFrame(
    {
        "상품명": X_test.reset_index(drop=True),
        "실제_분야": y_test.reset_index(drop=True),
        "예측_분야": y_pred,
        "일치_여부": ["O" if match else "X" for match in (y_test.values == y_pred)],
    }
)

print("=== 테스트 데이터 예측 및 일치 여부 (상위 20개) ===")
print(result.head(20))

Accuracy: 0.2167

=== Classification Report (Precision / Recall / F1-score) ===
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00         1
       경제/경영       0.00      0.00      0.00        11
          과학       0.00      0.00      0.00         2
          만화       0.00      0.00      0.00         2
          소설       0.19      1.00      0.32        11
       시/에세이       0.00      0.00      0.00         8
     어린이(초등)       0.00      0.00      0.00         5
       역사/문화       0.00      0.00      0.00         1
         외국어       1.00      0.50      0.67         2
          요리       0.00      0.00      0.00         1
          인문       1.00      0.12      0.22         8
        자기계발       0.00      0.00      0.00         4
       정치/사회       0.00      0.00      0.00         3
         청소년       0.00      0.00      0.00         1

    accuracy                           0.22        60
   macro avg       0.16      0.12      0.09        60


## 실습 13. Classification Report 확인하기

In [26]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 불러오기 및 전처리
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()

df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X, y 정의 및 train/test 분리 (Data Leakage 방지: split을 먼저 수행)
X = df_model["상품명"]
y = df_model["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42,
)

# 3. TF-IDF 변환 (train 데이터에만 fit 적용)
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. Naive Bayes 모델 학습 및 예측
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

# 5. [실습 12] Accuracy 출력
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")

# 6. [실습 13] Classification Report 확인하기 (Precision, Recall, F1-score)
report = classification_report(
    y_test,
    y_pred,
    zero_division=0,
)
print("=== Classification Report ===")
print(report)

# 7. 정답/예측/일치 여부(O/X) 표 작성 및 상위 20개 출력
result = pd.DataFrame(
    {
        "상품명": X_test.reset_index(drop=True),
        "실제_분야": y_test.reset_index(drop=True),
        "예측_분야": y_pred,
        "일치_여부": ["O" if match else "X" for match in (y_test.values == y_pred)],
    }
)

print("=== 테스트 데이터 예측 및 일치 여부 (상위 20개) ===")
print(result.head(20))

Accuracy: 0.2167

=== Classification Report ===
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00         1
       경제/경영       0.00      0.00      0.00        11
          과학       0.00      0.00      0.00         2
          만화       0.00      0.00      0.00         2
          소설       0.19      1.00      0.32        11
       시/에세이       0.00      0.00      0.00         8
     어린이(초등)       0.00      0.00      0.00         5
       역사/문화       0.00      0.00      0.00         1
         외국어       1.00      0.50      0.67         2
          요리       0.00      0.00      0.00         1
          인문       1.00      0.12      0.22         8
        자기계발       0.00      0.00      0.00         4
       정치/사회       0.00      0.00      0.00         3
         청소년       0.00      0.00      0.00         1

    accuracy                           0.22        60
   macro avg       0.16      0.12      0.09        60
weighted avg       0.20      0.2

## 실습 14. Confusion Matrix 확인하기

In [27]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 불러오기 및 전처리
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()

df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X, y 정의 및 train/test 분리 (stratify 사용 안 함)
X = df_model["상품명"]
y = df_model["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 3. TF-IDF 변환
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. Naive Bayes 모델 학습 및 예측
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

# 5. [실습 14] Confusion Matrix 확인하기
labels = model.classes_
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels,
)

df_cm = pd.DataFrame(
    cm,
    index=labels,
    columns=labels,
)

print("=== [실습 14] Confusion Matrix ===")
print(df_cm)
print("\n" + "=" * 50 + "\n")

# 6. 예측 결과 및 일치 여부(O/X) 표 출력 (상위 20개)
result = pd.DataFrame(
    {
        "상품명": X_test.reset_index(drop=True),
        "실제_분야": y_test.reset_index(drop=True),
        "예측_분야": y_pred,
        "일치_여부": ["O" if match else "X" for match in (y_test.values == y_pred)],
    }
)

print("=== 테스트 데이터 예측 및 일치 여부 (상위 20개) ===")
print(result.head(20))

=== [실습 14] Confusion Matrix ===
         가정/육아  경제/경영  과학  소설  시/에세이  어린이(초등)  역사/문화  예술/대중문화  외국어  요리  인문  \
가정/육아        0      0   0   1      0        0      0        0    0   0   0   
경제/경영        0      0   0  11      0        0      0        0    0   0   0   
과학           0      0   0   2      0        0      0        0    0   0   0   
소설           0      0   0  11      0        0      0        0    0   0   0   
시/에세이        0      0   0   8      0        0      0        0    0   0   0   
어린이(초등)      0      0   0   5      0        0      0        0    0   0   0   
역사/문화        0      0   0   1      0        0      0        0    0   0   0   
예술/대중문화      0      0   0   0      0        0      0        0    0   0   0   
외국어          0      0   0   1      0        0      0        0    1   0   0   
요리           0      0   0   1      0        0      0        0    0   0   0   
인문           0      0   0   7      0        0      0        0    0   0   1   
자기계발         0      0   0   4  

## 실습 15. 오분류 사례 확인하기

In [28]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 불러오기 및 전처리
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()

df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X, y 정의 및 train/test 분리 (stratify 제외)
X = df_model["상품명"]
y = df_model["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 3. TF-IDF 변환 (Data Leakage 방지: train만 fit)
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. Naive Bayes 모델 학습 및 예측
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

# 5. 전체 예측 결과 표 만들기
result = pd.DataFrame(
    {
        "상품명": X_test.reset_index(drop=True),
        "실제_분야": y_test.reset_index(drop=True),
        "예측_분야": y_pred,
    }
)

# 6. [실습 15] 오분류 사례 추출 및 확인하기
misclassified = result[
    result["실제_분야"] != result["예측_분야"]
].copy()

print("=== [실습 15] 오분류 사례 확인 ===")
print("오분류 수:", len(misclassified))
print("\n[오분류 데이터 상위 20개]")
print(misclassified.head(20))

=== [실습 15] 오분류 사례 확인 ===
오분류 수: 47

[오분류 데이터 상위 20개]
                          상품명    실제_분야 예측_분야
0    나는 미국 월배당 ETF로 40대에 은퇴한다    경제/경영    소설
1                        위버멘쉬       인문    소설
2   부자 아빠 가난한 아빠(20주년 특별 기념판)    경제/경영    소설
5                      월가의 영웅    경제/경영    소설
6                         긴긴밤  어린이(초등)    소설
7        영어로 문장 만들기 훈련 1차 임계점      외국어    소설
8                      이어령의 말       인문    소설
9           세상은 실제로 어떻게 돌아가는가       인문    소설
10                         책문    역사/문화    소설
11                    에그박사 14  어린이(초등)    소설
12             시대예보: 경량문명의 탄생    경제/경영    소설
14                      일의 감각     자기계발    소설
17                       코스모스       과학    소설
18              이로운 보수 의로운 진보    정치/사회    소설
19      나는 메트로폴리탄 미술관의 경비원입니다    시/에세이    소설
21                   워런 버핏 웨이    경제/경영    소설
22          무의식은 어떻게 나를 설계하는가       과학    소설
24         단 3개의 미국 ETF로 은퇴하라    경제/경영    소설
25             너에게 들려주는 단단한 말      청소년    소설
26           제로 투 원(10주년 기념판)    경제/경영    소설


## 실습 16. 예측 결과 저장하기

In [30]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. 데이터 불러오기 및 전처리
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

df_model = df_books[["상품명", "분야"]].copy()
for col in ["상품명", "분야"]:
    df_model[col] = df_model[col].fillna("").astype(str).str.strip()

df_model = df_model[
    (df_model["상품명"] != "") & (df_model["분야"] != "")
].reset_index(drop=True)

# 2. X, y 정의 및 train/test 분리 (stratify 제외)
X = df_model["상품명"]
y = df_model["분야"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 3. TF-IDF 변환
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. Naive Bayes 모델 학습
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 5. [실습 17] 새로운 도서 제목 예측하기
new_titles = [
    "파이썬으로 시작하는 데이터 분석",
    "처음 배우는 주식 투자",
    "마음을 이해하는 심리학",
]

# 가장 중요한 핵심: 새 제목에는 fit_transform이 아닌 transform()만 사용
new_vectors = tfidf.transform(new_titles)
new_predictions = model.predict(new_vectors)

new_result = pd.DataFrame(
    {
        "상품명": new_titles,
        "예상_분야": new_predictions,
    }
)

print("=== [실습 17] 새로운 도서 제목 예측 결과 ===")
print(new_result)

=== [실습 17] 새로운 도서 제목 예측 결과 ===
                 상품명 예상_분야
0  파이썬으로 시작하는 데이터 분석    소설
1       처음 배우는 주식 투자    소설
2       마음을 이해하는 심리학    소설
